[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/125/blob/main/06_series/06_2_Whole_Columns.ipynb)

# 06.2: Working on a Whole Column

In notebook 06.1 you created a series and looked up values one at a time. Most questions about data involve every value, though, such as "which majors pay more than $50,000?" With a list you would write a loop for that. This notebook shows how a series lets you do math, comparisons, filtering and sorting on all of its values at once, and how to change values safely.

This notebook opens in Google Colab. If Colab warns that the notebook was not authored by Google, click **Run anyway**. Your changes are not kept when you close the tab unless you choose **File > Save a copy in Drive**, so do that if you want to keep your work.

## The data

This uses the same majors data as 06.1, with three new series, all with the major names as their index. `share_women` holds the fraction of graduates in each major who are women, from 0 to 1. `employed` and `unemployed` hold the number of graduates in each group.

In [ ]:
import pandas as pd

majors = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_recent_grads.csv")
major_list = majors["Major"].to_list()
salaries = pd.Series(majors["Median"].to_list(), index=major_list)
share_women = pd.Series(majors["ShareWomen"].to_list(), index=major_list)
employed = pd.Series(majors["Employed"].to_list(), index=major_list)
unemployed = pd.Series(majors["Unemployed"].to_list(), index=major_list)
share_women.head(3)

The share for PETROLEUM ENGINEERING is about 0.12, so about 12 percent of those graduates are women. The dtype is `float64` because these numbers have decimal places.

## Math without a loop

Suppose you want the salaries in thousands of dollars. With a list, you would write a loop and build a new list. Here `.to_list()` turns the series back into an ordinary list.

In [ ]:
in_thousands_list = []
for salary in salaries.to_list():
    in_thousands_list.append(salary / 1000)
in_thousands_list[:3]

With a series, you divide the whole series by 1000. pandas applies the operation to every value and keeps every label. This is called a **vectorized** operation. It is shorter to write than the loop, and it also runs faster, which matters when a series has millions of values. So if you can do something to a series without a loop, you should.

In [ ]:
salaries / 1000

PETROLEUM ENGINEERING became 110.0 and LIBRARY SCIENCE became 22.0. The same idea works for the other arithmetic operators, so `salaries + 1000` adds 1000 to every value. It also works on text, where `+` joins strings just as it does for a single string.

In [ ]:
fruit_names = pd.Series(["apple", "banana"])
fruit_names + " is a fruit"

Each value now reads `apple is a fruit` or `banana is a fruit`.

## Results are new series

An operation like this does not change the original series. It produces a new series, which is displayed and then lost unless you assign it to a variable. The next cell shows that `salaries` still holds the original numbers.

In [ ]:
in_thousands = salaries / 1000
print(in_thousands.iloc[0])
print(salaries.iloc[0])

`in_thousands` starts with 110.0, and `salaries` still starts with 110000. The same is true of `.sort_values()` from 06.1: it gives you a new sorted series and leaves the original in its old order. To keep a result, assign it to a name. You can also assign it back to the original name, as in `salaries = salaries / 1000`, which replaces what the name refers to.

## Math between two series

You can also do math between two series. What fraction of the people in the labor force for each major are unemployed? That is the number unemployed divided by the number employed plus the number unemployed, and you can write it the same way.

In [ ]:
unemployment_rate = unemployed / (employed + unemployed)
unemployment_rate.loc[["NURSING", "PETROLEUM ENGINEERING"]]

For NURSING, this calculates 8497 / (180903 + 8497), or about 0.045, so about 4.5 percent. pandas did this separately for each of the 173 majors. One major, MILITARY TECHNOLOGIES, comes out as `NaN`, which marks a missing value, because it has 0 in both columns and 0 divided by 0 has no answer.

Something important happens when you combine two series: pandas lines the values up by **label**, not by position. A list has no labels, so it could only match by position. Here are two series with the names in a different order.

In [ ]:
paid = pd.Series([7.25, 71.28, 7.92], index=["Allen", "Braund", "Cumings"])
refund = pd.Series([2.00, 5.00, 1.00], index=["Cumings", "Allen", "Braund"])
paid - refund

Read the result by label. Allen's value is 7.25 - 5.00 = 2.25, even though Allen is first in one series and second in the other. This matching is called **alignment**, and it means that you do not have to sort two series before combining them. If a label appears in only one of the series, pandas has nothing to pair it with, so the result is `NaN` for that label.

In [ ]:
one_refund = pd.Series([5.00, 1.00], index=["Allen", "Davies"])
paid - one_refund

Only Allen appears in both series, so only Allen gets a number. Braund, Cumings and Davies come out as `NaN`. If you see unexpected `NaN` values after combining two series, check whether their labels match.

## Comparisons and boolean series

A comparison also works on every value, and it produces a series of `True` and `False` values with the same labels.

In [ ]:
salaries > 70000

A series of booleans like this one is called a **boolean mask**. On its own it is hard to read, since only a few of the 173 values are `True`. There are two useful things to do with it. The first is to count: `True` counts as 1 and `False` counts as 0, so `.sum()` gives the number of `True` values.

In [ ]:
(salaries > 70000).sum()

Three majors have a median above 70000. The second is to filter: when you put a mask inside the square brackets, pandas keeps only the items where the mask is `True`.

In [ ]:
salaries[salaries > 70000]

Those three are PETROLEUM ENGINEERING (110000), MINING AND MINERAL ENGINEERING (75000) and METALLURGICAL ENGINEERING (73000). Read `salaries[salaries > 70000]` from the inside out: first build the mask, then use it to select from the series. The result is a new series, so `salaries` itself still has all 173 items. You can also put the mask inside `.loc`, as in `salaries.loc[salaries > 70000]`, which gives the same result.

Doing this in two steps, with a name for the mask, makes your code easier to read and to fix when something goes wrong.

In [ ]:
high_pay = salaries > 70000
salaries[high_pay]

This gives the same three majors. Watch the difference between `>` and `>=`. In this data 17 majors have a median of exactly 40000, so `salaries > 40000` and `salaries >= 40000` give different answers.

In [ ]:
print((salaries > 40000).sum())
print((salaries >= 40000).sum())

## Combining conditions

Because the two series have the same index, a mask built from one can be used to filter the other, and masks can be combined. Use `&` for and, `|` for or, and `~` for not. Each condition needs its own parentheses, and the Python words `and`, `or` and `not` will not work here, because they expect a single `True` or `False` and not a whole series of them.

Which majors have a median above $50,000 and are mostly women?

In [ ]:
salaries[(salaries > 50000) & (share_women > 0.5)]

Only ASTRONOMY AND ASTROPHYSICS, at 62000, meets both conditions, out of the 25 majors above $50,000 and the 96 majors that are mostly women. Now an example with `|`: which majors have a median that is either above $70,000 or below $25,000?

In [ ]:
salaries[(salaries > 70000) | (salaries < 25000)]

This returns the three majors from before and also COUNSELING PSYCHOLOGY (23400) and LIBRARY SCIENCE (22000). `~` flips a mask, turning every `True` into `False` and the reverse.

In [ ]:
(~(share_women > 0.5)).sum()

That gives 77 majors, which together with the 96 above makes all 173. One of the 77 is FOOD SCIENCE, whose share is missing from the data. A `>` comparison with a missing value gives `False`, so `~` turns it into `True`, and FOOD SCIENCE gets counted even though we do not know its share. Notebook 06.3 returns to missing values.

To check whether values match any item in a list, use `.isin()`, which is shorter than chaining many `|` conditions. It also makes a mask. The index has an `.isin()` of its own, which is useful for selecting or leaving out particular labels.

In [ ]:
print(salaries.isin([110000, 75000]).sum())
salaries[salaries.index.isin(["NURSING", "ECONOMICS"])]

Two values match the first list, 110000 and 75000. The second line keeps the items labeled NURSING and ECONOMICS. To leave those labels out instead, you would put `~` in front of the mask, as in `salaries[~salaries.index.isin(["NURSING", "ECONOMICS"])]`.

## Sorting

`.sort_values()` sorts by value, and `.sort_index()` sorts by label. Both sort from low to high unless you pass `ascending=False`. Combining a descending sort with `.head()` gives you a top-N list.

In [ ]:
salaries.sort_values(ascending=False).head(3)

In [ ]:
salaries.sort_index().head(3)

The first result lists the three highest medians, and the second lists the first three majors alphabetically: ACCOUNTING, ACTUARIAL SCIENCE and ADVERTISING AND PUBLIC RELATIONS. Chaining like this works because `.sort_values()` returns a series, and you can call `.head()` on that series. `.nlargest()` and `.nsmallest()` are shortcuts for this common job.

In [ ]:
salaries.nsmallest(2)

The two smallest are LIBRARY SCIENCE (22000) and COUNSELING PSYCHOLOGY (23400). Remember that none of these sorts change `salaries`. Sorting methods do have an `inplace=True` option that changes the series directly, like the list method `.sort()`, but we will not use it. Assigning the result to a name makes it clearer what your code is doing.

## Copies and aliases

Assigning a series to another name does not copy it, the same as with lists. Both names refer to one object, which is called an **alias**. To get an independent series, use `.copy()`.

In [ ]:
alias = salaries
independent = salaries.copy()
print(alias is salaries)
print(independent is salaries)

`alias` is the same object as `salaries`, and `independent` is a different one. The difference shows up as soon as you change a value, which is the subject of the next section.

## Changing values

Unlike the operations above, assigning with `.loc` or `.iloc` changes the series directly. Here we work on a copy so that `salaries` stays as it was.

In [ ]:
updated = salaries.copy()
updated.loc["NURSING"] = 50000
print(updated.loc["NURSING"])
print(salaries.loc["NURSING"])

The copy now has 50000 and `salaries` still has 48000. If we had written `updated = salaries`, without `.copy()`, then the change would have shown up under both names. You can also change many values at once by putting a mask inside `.loc`. The next cell raises every salary below 25000 to 25000 in a copy.

In [ ]:
floored = salaries.copy()
floored.loc[floored < 25000] = 25000
floored.tail(3)

COUNSELING PSYCHOLOGY and LIBRARY SCIENCE are now both 25000, and CLINICAL PSYCHOLOGY stays at 25000 because it was not below the cutoff. A slice works too, either with one value for all of the items or with a list of new values. If a label is repeated, assigning to that label changes all of its values.

In [ ]:
fruit = pd.Series([180, 120, 15, 650], index=["apple", "apple", "cherry", "date"])
fruit.loc["apple"] = 333
fruit.iloc[2:] = [20, 700]
fruit

Both apples are now 333, and the slice replaced cherry and date with 20 and 700.

## Changing the labels

The index does not allow you to change one label on its own, but you can replace all of the labels at once by assigning a list of the same length to `.index`.

In [ ]:
fruit.index = ["red apple", "green apple", "cherry", "date"]
fruit

The values did not change, and the four items now have the new labels.

## Putting it together: state populations

Here is a worked example that uses these tools, with estimated 2019 populations for the U.S. states and D.C.

In [ ]:
# lists of state names and populations
state_list = ['California', 'Texas', 'Florida', 'New York', 'Illinois',
              'Pennsylvania', 'Ohio', 'Georgia', 'North Carolina', 'Michigan',
              'New Jersey', 'Virginia', 'Washington', 'Arizona', 'Massachusetts',
              'Tennessee', 'Indiana', 'Missouri', 'Maryland', 'Wisconsin',
              'Colorado', 'Minnesota', 'South Carolina', 'Alabama', 'Louisiana',
              'Kentucky', 'Oregon', 'Oklahoma', 'Connecticut', 'Utah', 'Iowa',
              'Nevada', 'Arkansas', 'Mississippi', 'Kansas', 'New Mexico',
              'Nebraska', 'West Virginia', 'Idaho', 'Hawaii', 'New Hampshire',
              'Maine', 'Montana', 'Rhode Island', 'Delaware', 'South Dakota',
              'North Dakota', 'Alaska', 'DC', 'Vermont', 'Wyoming']

population_list = [39512223, 28995881, 21477737, 19453561, 12671821, 12801989,
                   11689100, 10617423, 10488084, 9986857, 8882190, 8535519,
                   7614893, 7278717, 6949503, 6833174, 6732219, 6137428,
                   6045680, 5822434, 5758736, 5639632, 5148714, 4903185,
                   4648794, 4467673, 4217737, 3956971, 3565287, 3205958,
                   3155070, 3080156, 3017825, 2976149, 2913314, 2096829,
                   1934408, 1792147, 1787065, 1415872, 1359711, 1344212,
                   1068778, 1059361, 973764, 884659, 762062, 731545, 705749,
                   623989, 578759,]


state_series = pd.Series(population_list, index=state_list)
state_series.head(3)

California leads with 39512223, followed by Texas and Florida. `.sum()` adds up the values, which gives a total U.S. population, and dividing by that total turns every value into a fraction of the whole. Multiplying by 100 makes it a percentage, and `.round(2)` keeps two decimal places.

In [ ]:
total_population = state_series.sum()
print(total_population)
state_percent = (state_series / total_population * 100).round(2)
state_percent.head(3)

The total is 328300544, and California has 12.04 percent of it, Texas 8.83 percent and Florida 6.54 percent. Now a question that needs a loop after all: how few of the largest states account for more than half of the population? A loop is the right tool here, because each step depends on the running total and we want to stop partway through.

In [ ]:
state_percent = state_percent.sort_values(ascending=False)
top_states = []
percent_sum = 0

for state in state_percent.index:
    top_states.append(state)
    percent_sum = percent_sum + state_percent.loc[state]
    if percent_sum >= 50:
        break

print(len(top_states), "states have", round(percent_sum, 2), "percent of the population:")
print(top_states)

Nine states, from California through North Carolina, have about 51.08 percent of the population. Notice that the sort moved Pennsylvania ahead of Illinois, which the original list had in the wrong order. The loop goes through `state_percent.index`, and it uses each label to look up the value with `.loc`.

## Practice problems

These problems are not collected or graded. They are here so that you can find out whether you can write this code yourself before the reading check and the pair activity. Write your answer in the cell under each problem, run it, and then run the check cell below it, which tells you whether your answer is right. The check cells are collapsed because they contain the answers, so try the problem before you open one.

Run the next two cells first. The first one creates fresh data for the problems, and the second sets up the checks.

In [ ]:
import pandas as pd

majors = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_recent_grads.csv")
major_list = majors["Major"].to_list()
salaries = pd.Series(majors["Median"].to_list(), index=major_list)
share_women = pd.Series(majors["ShareWomen"].to_list(), index=major_list)
full_time = pd.Series(majors["Full_time"].to_list(), index=major_list)
part_time = pd.Series(majors["Part_time"].to_list(), index=major_list)
salaries_start = salaries.copy()
full_time_start = full_time.copy()
part_time_start = part_time.copy()
share_women_start = share_women.copy()

In [ ]:
#@title Run this cell to set up the checks
import numpy as np
import pandas as pd


def same(got, want):
    """True when got matches want: same labels and values for a series, same value for a number."""
    try:
        if isinstance(want, pd.Series):
            if not isinstance(got, pd.Series) or list(got.index) != list(want.index):
                return False
            if pd.api.types.is_numeric_dtype(want):
                return bool(np.allclose(got.astype(float), want.astype(float), equal_nan=True))
            return list(got) == list(want)
        if isinstance(want, (list, np.ndarray)):
            return list(got) == list(want)
        if isinstance(want, str):
            return got == want
        if isinstance(got, (pd.Series, pd.DataFrame, list, np.ndarray)):
            return False
        return bool(np.isclose(float(got), float(want)))
    except (TypeError, ValueError):
        return False


def check(number, name, want, hint, unchanged=None):
    """Print feedback for one problem. unchanged is (variable name, its starting value)."""
    if name not in globals():
        print(f"Problem {number}: not yet. No variable named {name} exists. Did you run your answer cell?")
    elif not same(globals()[name], want):
        print(f"Problem {number}: not yet. {hint}")
    elif unchanged is not None and not globals()[unchanged[0]].equals(unchanged[1]):
        print(f"Problem {number}: {name} is right, but {unchanged[0]} changed. Work on a copy, then run the setup cell again.")
    else:
        print(f"Problem {number}: correct.")

### Problem 1
Create a series named `monthly` that gives each major's median salary per month, which is `salaries` divided by 12. `salaries` itself should not change.

In [ ]:
# Problem 1

In [ ]:
#@title Check Problem 1
check(1, "monthly", salaries_start / 12, "Divide the whole series by 12 and assign the result to monthly.", ("salaries", salaries_start))

### Problem 2
`full_time` and `part_time` hold the number of graduates working full time and part time in each major. Create a series named `part_time_share` that gives, for each major, the part-time number divided by the sum of the two.

In [ ]:
# Problem 2

In [ ]:
#@title Check Problem 2
check(2, "part_time_share", part_time_start / (full_time_start + part_time_start), "Use math between series, with parentheses around the sum.")

### Problem 3
Store the number of majors with a median salary of at least 60000 in a variable named `count_high_pay`. A median of exactly 60000 counts.

In [ ]:
# Problem 3

In [ ]:
#@title Check Problem 3
check(3, "count_high_pay", (salaries_start >= 60000).sum(), "Make a mask with >= and then use .sum() on it.")

### Problem 4
Create a series named `middle_pay` with only the items of `salaries` from 40000 through 50000, including both 40000 and 50000.

In [ ]:
# Problem 4

In [ ]:
#@title Check Problem 4
check(4, "middle_pay", salaries_start[(salaries_start >= 40000) & (salaries_start <= 50000)], "Combine two conditions with &. Each one needs its own parentheses, and both ends are included.")

### Problem 5
Create a series named `top_five` with the five highest values of `share_women`, ordered from highest to lowest.

In [ ]:
# Problem 5

In [ ]:
#@title Check Problem 5
check(5, "top_five", share_women_start.sort_values(ascending=False).head(5), "Sort from high to low and then take the first five.")

### Problem 6
Create a copy of `salaries` named `adjusted`, and in the copy change the value for `'MUSIC'` to 36000. `salaries` itself should not change.

In [ ]:
# Problem 6

In [ ]:
#@title Check Problem 6
adjusted_want = salaries_start.copy()
adjusted_want.loc["MUSIC"] = 36000
check(6, "adjusted", adjusted_want, "Use .copy() and then assign with .loc.", ("salaries", salaries_start))

### Problem 7
Use `.index.isin()` to create a series named `without_two` that has every item of `salaries` except the ones labeled `'MUSIC'` and `'NURSING'`.

In [ ]:
# Problem 7

In [ ]:
#@title Check Problem 7
check(7, "without_two", salaries_start[~salaries_start.index.isin(["MUSIC", "NURSING"])], "Build the mask with salaries.index.isin() and flip it with ~.")

## What's next

You can now do math on a whole series, filter it with masks, sort it and change it safely. So far you have mostly picked out individual items, but often you want a summary of the whole column, such as what a typical value is and how spread out the values are. Notebook 06.3 introduces the descriptive statistics that answer those questions.